In this notebook we'll explore different document loaders available through `langchain`.

## PyPDF loader

This is suited to simple, clean PDFs.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
doc1_path = "/Users/ashapatel/Documents/projects/rag_cc/documents/insurance-product-information-document.pdf"

doc2_path = "/Users/ashapatel/Documents/projects/rag_cc/documents/policy-wording.pdf"

In [ ]:
loader1 = PyPDFLoader(doc1_path)
document1 = loader1.load()

In [ ]:
loader2 = PyPDFLoader(doc2_path)
document2 = loader2.load()

### Explore the extracted page content

In [ ]:
for page_num, page in enumerate(document1):
    print(f"=== Page {page_num + 1} ===")
    print(page.page_content)
    print()

The first page in document1 contains 2 columns each containing bullet points. This document loader correctly extracted each bullet point, reading down the left-hand column then the right-hand column.

In [ ]:
for page_num, page in enumerate(document2):
    print(f"=== Page {page_num + 1} ===")
    print(page.page_content)
    print()

The second document looks good too. The 4-column table in the appendix was extracted row-by-row.

One issue I noticed is that some bullet points have been extracted as the letter 'Y'.

### Cleaning page content

Looking at the page content, there's a lot of whitespace e.g between section headings and the text in the given section. We could remove this whitespace so that the section heading and text are more likely to appear in the same chunk, which helps preserve context and also reduces token usage.

Let's write a function to clean the page content:
- Collapse whitespace, i.e multiple spaces or tabs back-to-back
- Wherever 3 or more new lines occur consecutively, replace them with 2 new lines

The new line processing is informed by the fact that LangChain's `RecursiveCharacterTextSplitter` -- which we'll use for chunking -- looks for `\n\n` as the plain-text representation of a paragraph break.

In [ ]:
import re


def clean_text(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


assert clean_text(" Hello   World ") == "Hello World"
assert clean_text("Hello\n\n\nWorld") == "Hello\n\nWorld"

Let's test this cleaning function on a page from our first document.

In [ ]:
doc1_page_content = documents[0].page_content
print(doc1_page_content)

In [ ]:
cleaned_content = clean_text(doc1_page_content)
print(cleaned_content)

Comparing this to the original page content before cleaning, we can see that the large sections of whitespace have been removed. 

Now let's test the cleaning function on some pages from our second document, which is much longer (84 pages) and has more variety in page layout.

In [ ]:
loader = PyPDFLoader(doc2_path)
documents = loader.load()

doc2_page_content = documents[1].page_content
print(doc2_page_content)

In [ ]:
cleaned_content = clean_text(doc2_page_content)
print(cleaned_content)

### Explore the extracted metadata

In [ ]:
for page_num, page in enumerate(documents):
    print(f"=== Page {page_num + 1} ===")
    print(page.metadata)
    print()

Looking at the metadata, there are a few things that would be useful to persist to the chunking step:
- `source`: currently this gives the whole file path, but we're just interested in the file name
- `page`: this will enable citations referencing the specific page number on which the chunk appears in the original document

We're not interested in metadata such as `producer`, `creator` and `moddate` as they don't relate to the document's content.

## Directory loader

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

In [ ]:
directory_path = "/Users/ashapatel/Documents/projects/rag_cc/documents/"

loader = DirectoryLoader(
    directory_path, glob="**/*.pdf", loader_cls=PyPDFLoader, show_progress=True
)
documents = loader.load()

print(f"Number of Documents: {len(documents)}")

for idx, document in enumerate(documents, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

This loads all documents in a given directory. This could be useful for my pipeline, but to start with I think I'll specify each document path separately.

# Classify page elements

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader

This loader is suited to PDFs with complex layouts, or scanned documents which require OCR to extract the text. It classifies page elements such as titles, tables, paragraphs etc.

In [ ]:
loader1 = UnstructuredPDFLoader(doc1_path)
documents1 = loader1.load()

print(f"Number of Documents: {len(documents1)}")

for idx, document in enumerate(documents1, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

In [ ]:
loader2 = UnstructuredPDFLoader(doc2_path)
documents2 = loader2.load()

print(f"Number of Documents: {len(documents2)}")

for idx, document in enumerate(documents2, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

For the first document, this loader mixes up the order of bullet points from the left-hand and right-hand columns, i.e doesn't read down the left-hand column first like the basic PyPDFLoader does.

For the second document, like PyPDFLoader this loader extracts some bullet points as the letter 'Y', but correctly reads the appendix table row by row.

Even if this loader did work better than simpler ones, we'd need to consider the extra processing time it takes.